# [LAB] LangGraph 심화 실습

그래프를 나누고(Subgraph), 상태를 저장하고(Checkpointer, Memory Store), 여러 작업을 동시에 실행하는(Fan-out/Fan-in, Send) 코드를 완성하고 실행해 봅니다. 보기 중 하나를 골라 주석을 풀어 코드를 완성하고, 완성된 코드는 실행하며 값을 바꿔 봅니다.

- **2장 Subgraph로 요약·번역 나누기**: 요약·번역 그래프를 따로 만들어 부모 그래프의 노드 안에서 호출합니다.
- **3장 MemorySaver로 대화 이어 가기**: 체크포인터를 붙여 앞에서 한 말을 기억하게 합니다.
- **4장 SqliteSaver로 저장하고 되돌아가기**: 대화 기록을 파일에 남기고, 지난 시점으로 되돌아가 다시 실행합니다.
- **5장 Memory Store로 대화를 넘어 기억하기**: 새 대화에서도 사용자의 이름과 취미를 기억하게 합니다.
- **6장 장점과 단점을 동시에 정리하기**: 두 노드를 동시에 실행하고 한 노드에서 합칩니다.
- **7장 질문 수만큼 동시에 답하기**: 질문 목록 길이만큼 작업을 만들어 동시에 실행합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 모든 장이 LLM을 호출하므로 결과가 나오기까지 수십 초가 걸릴 수 있고, 답은 실행할 때마다 조금씩 다릅니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. Subgraph로 요약·번역 나누기
  - 2.1 요약·번역 Subgraph
  - 2.2 부모 그래프의 State와 노드
  - 2.3 부모 그래프 연결
  - 2.4 실행하기
- 3. MemorySaver로 대화 이어 가기
  - 3.1 State와 chatbot 노드
  - 3.2 그래프 연결
  - 3.3 MemorySaver로 컴파일하고 실행하기
- 4. SqliteSaver로 저장하고 되돌아가기
  - 4.1 SqliteSaver로 저장하기
  - 4.2 체크포인트 목록 보기
  - 4.3 과거 체크포인트로 되돌아가기
- 5. Memory Store로 대화를 넘어 기억하기
  - 5.1 장기 기억 Store 만들기
  - 5.2 기억을 읽고 쓰는 chatbot 노드
  - 5.3 그래프 연결과 컴파일
  - 5.4 대화 함수
  - 5.5 두 세션으로 대화하기
- 6. 장점과 단점을 동시에 정리하기
  - 6.1 장점·단점·결론 노드
  - 6.2 병렬로 연결하기
  - 6.3 실행하기
- 7. 질문 수만큼 동시에 답하기
  - 7.1 State와 worker 노드
  - 7.2 fan_out 함수
  - 7.3 동적으로 연결하기
  - 7.4 실행하기
- 8. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env()

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고칩니다. `.env`는 노트북을 시작할 때 한 번만 읽으므로, 고친 뒤에는 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/03-langgraph-advanced/` 네 파일. 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다. 네 파일 모두 raw를 기준으로 하고, 원본 속 이전 강사 메모 주석·주석 처리된 코드·디버그 출력은 셀에서 뺐습니다(3-4 목차 3절).
- `02_persistence.py`와 `04_parallel_execution.py`는 각각 두 장으로 나눴습니다. 뒤 장의 접힌 준비 코드가 앞 장의 정의(완성본)를 다시 담고 있어서, 앞 장 빈칸을 못 채웠어도 뒤 장은 실행됩니다.
- 과제로 바꾸지 않은 TODO(compile 한 줄, Store 생성·조회·저장, SqliteSaver compile)는 채운 채로 두고 TODO 표시만 지웠습니다.
- 사용자 결정(2026-09-28): 03 compile 줄은 슬라이드대로 patch하고 "왜 안 달라지나"를 CHANGE_HERE 2로 확인, 02는 DB를 노트북 폴더에 두고 체크포인트 목록을 번호와 함께 출력, 되돌아갈 번호를 CHANGE_HERE로. 04의 Send(`fan_out`·동적 연결)는 과제로 올렸습니다.
- 사외 실측(2026-09-28, `COURSE_ENV=out`, OpenAI): 빈칸을 정답으로 채운 사본을 새 커널로 실행해 오류 없이 끝났습니다(LLM 약 14회, 전체 약 25초). CHANGE_HERE를 바꾼 사본(체크포인트 5번, user_002, 질문 추가 / compile에서 store 제거)도 실행해 성공 기준대로 나오는 것을 확인했습니다: 5번(메시지 수 0)으로 되돌아가면 "아직 취미를 말한 적 없다", user_002는 "아직 몰라요", store를 빼도 결과 동일, 질문 4개면 4줄. 사내 게이트웨이 소요 시간은 미확인입니다.
- CodeMate가 노트북에서 선택한 셀을 읽는지, 셀을 직접 고칠 수 있는지는 강의장에서 아직 확인하지 않았습니다(미확인).

</details>

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- `# ✅ CHOOSE_HERE:` 아래에 보기가 주석으로 들어 있습니다. 줄 끝의 번호(1️⃣~4️⃣)와 설명을 보고 하나를 고릅니다. 같은 번호가 붙은 줄을 모두 선택하고 `Ctrl + /`를 눌러 주석을 푼 뒤 실행합니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. ⛔이면 푼 줄을 다시 선택하고 `Ctrl + /`로 주석 처리한 뒤 다른 보기를 고릅니다. 마지막으로 접힌 "정답 보기"에서 보기마다의 해설을 읽습니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. Subgraph로 요약·번역 나누기

문서를 받아 요약하고 영어로 옮긴 뒤, 그 요약을 바탕으로 질문에 답하는 그래프를 만듭니다. 요약·번역은 따로 만든 작은 그래프(Subgraph)가 맡고, 부모 그래프는 노드 하나에서 그 Subgraph를 호출합니다.

두 그래프의 State는 다릅니다. Subgraph는 `text`(원문)와 `summary`(요약)만 알고, 부모는 `messages`(대화), `document`(문서), `summary`(영문 요약)를 가집니다. 부모 노드가 값을 옮겨 넣고 받아 와야 합니다.

그래프는 맨 아래 "실행하기" 셀에서 처음 실행되므로, **보기를 모두 고른 뒤에** 결과를 볼 수 있습니다. 보기를 고른 셀은 그때마다 실행해 두고, 마지막 셀에서 한꺼번에 확인합니다.

준비 코드입니다. 필요한 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 접혀 있으니 펼치지 않아도 되고, 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- raw와 솔루션의 차이: 솔루션에는 긴 강사 메모 docstring, 주석 처리된 확장 구상 그림, 주석 처리된 서브그래프 단독 호출 코드가 있습니다. raw를 썼고, raw의 메모 주석("따라간다ㅓ")은 뺐습니다.
- 44행·91행 compile TODO는 한 줄짜리라 빈칸으로 두지 않았습니다.
- LLM 호출 3회(요약, 번역, 답변).
- 원본은 마지막 답만 `print(...content)`로 출력합니다. 노트북은 결과 변수를 그래프가 드러나는 이름(`parent_graph_result`)으로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).

</details>

### 2.1 요약·번역 Subgraph

**✅ summarize, translate 노드와 START/END 엣지를 연결해 주세요.**

- 🟢 2.4의 `영문 요약:` 뒤에 영어 한 문장이 나온다
- ⛔ 2.4의 `영문 요약:` 뒤가 한국어이거나, 이 셀에서 `Graph must have an entrypoint` 오류가 난다 → 다른 보기를 고릅니다

In [ ]:
# ── 서브그래프 ────────────────────────────────────────────────────────────────
# 서브그래프는 독립적으로 컴파일된 그래프로, 부모 그래프의 노드로 사용된다.
# 복잡한 워크플로우를 재사용 가능한 단위로 분리할 때 활용한다.


class SummaryState(TypedDict):
    """서브그래프 전용 State. 부모 State와 독립적이다."""

    text: str
    summary: str


def summarize_node(state: SummaryState) -> dict:
    result = llm.invoke(f"다음 텍스트를 한 문장으로 요약해줘:\n\n{state['text']}")
    return {"summary": result.content}


def translate_node(state: SummaryState) -> dict:
    result = llm.invoke(f"다음 문장을 영어로 번역해줘:\n\n{state['summary']}")
    return {"summary": result.content}


summary_builder = StateGraph(SummaryState)
# ✅ CHOOSE_HERE: summarize, translate 노드와 START/END 엣지를 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# summary_builder.add_node("summarize", summarize_node)  # 1️⃣ 노드 두 개 + START→summarize→END
# summary_builder.add_node("translate", translate_node)  # 1️⃣
# summary_builder.add_edge(START, "summarize")           # 1️⃣
# summary_builder.add_edge("summarize", END)             # 1️⃣
#
# summary_builder.add_node("summarize", summarize_node)  # 2️⃣ 노드 두 개 + START→translate→summarize→END
# summary_builder.add_node("translate", translate_node)  # 2️⃣
# summary_builder.add_edge(START, "translate")           # 2️⃣
# summary_builder.add_edge("translate", "summarize")     # 2️⃣
# summary_builder.add_edge("summarize", END)             # 2️⃣
#
# summary_builder.add_node("summarize", summarize_node)  # 3️⃣ 노드 두 개 + START→summarize→translate→END
# summary_builder.add_node("translate", translate_node)  # 3️⃣
# summary_builder.add_edge(START, "summarize")           # 3️⃣
# summary_builder.add_edge("summarize", "translate")     # 3️⃣
# summary_builder.add_edge("translate", END)             # 3️⃣
#
# summary_builder.add_node("summarize", summarize_node)  # 4️⃣ 노드 두 개 + summarize→translate→END
# summary_builder.add_node("translate", translate_node)  # 4️⃣
# summary_builder.add_edge("summarize", "translate")     # 4️⃣
# summary_builder.add_edge("translate", END)             # 4️⃣
# 서브그래프를 독립적으로 컴파일
summary_subgraph = summary_builder.compile()

<details><summary>정답 보기</summary>

Subgraph에 노드 두 개를 등록하고 차례로 잇습니다. `summarize`가 원문을 한 문장으로 요약하고, `translate`가 그 요약을 영어로 옮깁니다. 바로 아래 줄이 이 그래프를 따로 컴파일해 `summary_subgraph`로 만듭니다.

**정답: 3️⃣**

- 1️⃣ translate 노드를 등록만 하고 잇지 않았습니다. 오류 없이 실행되지만 번역을 건너뛰어 2.4의 `영문 요약:` 뒤에 한국어 요약이 나옵니다.
- 2️⃣ 번역을 먼저 하고 요약을 나중에 합니다. 번역할 요약이 아직 없고, 마지막 요약이 `summary`를 한국어로 덮어씁니다. 오류 없이 2.4의 `영문 요약:` 뒤에 한국어 문장이 나옵니다.
- 3️⃣ 두 노드를 등록하고 요약 → 번역 순서로 잇습니다. 요약한 한국어 문장을 번역 노드가 영어로 바꿉니다.
- 4️⃣ START에서 나가는 엣지가 없어 어디서 시작할지 모릅니다. 바로 아래 컴파일 줄에서 `Graph must have an entrypoint` 오류가 납니다.

</details>

### 2.2 부모 그래프의 State와 노드

**✅ summary_subgraph를 호출해 document를 요약/번역해 주세요.**

- 🟢 2.4의 `영문 요약:` 뒤에 문서 내용(LLM의 등장, AI가 하는 일)을 담은 영어 문장이 나온다
- ⛔ 2.4에서 `KeyError: 'text'`가 나거나 `영문 요약:` 뒤가 비어 있다 → 다른 보기를 고릅니다

In [ ]:
# ── 부모 그래프 ───────────────────────────────────────────────────────────────


class ParentState(TypedDict):
    messages: Annotated[list, add_messages]
    document: str
    summary: str


def fetch_document(state: ParentState) -> dict:
    # 실제로는 DB나 파일에서 문서를 가져온다.
    doc = "인공지능은 인간의 지능을 모방하는 컴퓨터 시스템으로, 최근 LLM의 등장으로 급격히 발전했습니다. 이제 AI는 텍스트 생성, 번역, 코딩 등 다양한 작업을 수행할 수 있습니다."
    return {"document": doc}


def summarize_document(state: ParentState) -> dict:
    # 서브그래프를 노드처럼 호출
    # ✅ CHOOSE_HERE: summary_subgraph를 호출해 document를 요약/번역해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # response = summary_subgraph.invoke({"text": state["document"], "summary": ""})  # 1️⃣ document를 text에 넣어 호출, summary를 돌려줌
    # return {"summary": response["summary"]}                 # 1️⃣
    #
    # response = summary_subgraph.invoke({"document": state["document"], "summary": ""})  # 2️⃣ document 키 그대로 호출
    # return {"summary": response["summary"]}                 # 2️⃣
    #
    # response = summary_subgraph.invoke({"text": state["document"], "summary": ""})  # 3️⃣ 결과를 document에 담아 돌려줌
    # return {"document": response["summary"]}                # 3️⃣
    #
    # response = summary_subgraph.invoke(state["document"])   # 4️⃣ 문서 글자를 그대로 넘겨 호출
    # return {"summary": response["summary"]}                 # 4️⃣
    pass  # 그대로 둡니다


def respond_node(state: ParentState) -> dict:
    response = llm.invoke(
        f"문서 요약(영문): {state['summary']}\n\n이를 바탕으로 사용자 질문에 답해줘: {state['messages'][-1].content}"
    )
    return {"messages": [response]}

<details><summary>정답 보기</summary>

`summarize_document`는 부모 그래프의 노드입니다. 부모 State의 `document`를 Subgraph State의 `text`에 넣어 `summary_subgraph`를 호출하고, Subgraph가 돌려준 `summary`를 부모 State에 담아 돌려줘야 합니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다.

**정답: 1️⃣**

- 1️⃣ 부모의 `document`를 Subgraph State의 `text` 자리에 넣어 호출하고, Subgraph가 만든 `summary`를 부모 State의 `summary`에 담아 돌려줍니다.
- 2️⃣ Subgraph State에는 `document`가 없고 `text`가 있습니다. 요약 노드가 `text`를 찾지 못해 2.4에서 `KeyError: 'text'`가 납니다.
- 3️⃣ 요약을 `document`에 덮어써서 부모의 `summary`는 빈 채로 남습니다. 오류 없이 2.4의 `영문 요약:` 뒤가 비고, `Ai Message` 칸의 답은 요약할 문서가 없다고 말합니다.
- 4️⃣ 그래프는 State 모양의 딕셔너리를 받아야 합니다. 문서 글자만 넘겨 2.4에서 `InvalidUpdateError: Expected dict` 오류가 납니다.

</details>

### 2.3 부모 그래프 연결

**✅ 부모 그래프의 실행 순서를 START → fetch_document → summarize_document → respond → END로 연결해 주세요.**

- 🟢 2.4에서 `영문 요약:` 줄과 `Ai Message` 칸이 모두 나온다
- ⛔ 이 셀에서 `Graph must have an entrypoint` 오류가 나거나, 2.4의 답이 문서가 없다고 말한다 → 다른 보기를 고릅니다

In [ ]:
parent_builder = StateGraph(ParentState)
parent_builder.add_node("fetch_document", fetch_document)
parent_builder.add_node("summarize_document", summarize_document)  # 서브그래프 노드
parent_builder.add_node("respond", respond_node)

# ✅ CHOOSE_HERE: 부모 그래프의 실행 순서를 START → fetch_document → summarize_document → respond → END로 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# parent_builder.add_edge(START, "fetch_document")            # 1️⃣ fetch_document → respond로 바로 잇기
# parent_builder.add_edge("fetch_document", "respond")        # 1️⃣
# parent_builder.add_edge("respond", END)                     # 1️⃣
#
# parent_builder.add_edge("fetch_document", "summarize_document")  # 2️⃣ START 엣지 없이 세 노드 잇기
# parent_builder.add_edge("summarize_document", "respond")    # 2️⃣
# parent_builder.add_edge("respond", END)                     # 2️⃣
#
# parent_builder.add_edge(START, "summarize_document")        # 3️⃣ 요약을 먼저, 문서 가져오기를 나중에
# parent_builder.add_edge("summarize_document", "fetch_document")  # 3️⃣
# parent_builder.add_edge("fetch_document", "respond")        # 3️⃣
# parent_builder.add_edge("respond", END)                     # 3️⃣
#
# parent_builder.add_edge(START, "fetch_document")            # 4️⃣ START→fetch_document→summarize_document→respond→END
# parent_builder.add_edge("fetch_document", "summarize_document")  # 4️⃣
# parent_builder.add_edge("summarize_document", "respond")    # 4️⃣
# parent_builder.add_edge("respond", END)                     # 4️⃣

app = parent_builder.compile()

<details><summary>정답 보기</summary>

부모 그래프의 노드 세 개는 등록되어 있습니다. 문서를 가져오고(`fetch_document`), Subgraph로 요약·번역하고(`summarize_document`), 요약을 바탕으로 답하는(`respond`) 순서로 잇습니다. 보기 아래의 `app = parent_builder.compile()` 줄이 부모 그래프를 컴파일합니다.

**정답: 4️⃣**

- 1️⃣ 요약 노드를 건너뜁니다. 오류 없이 2.4의 `영문 요약:` 뒤가 비고, `Ai Message` 칸의 답은 요약할 문서가 없다고 말합니다.
- 2️⃣ START에서 나가는 엣지가 없습니다. 이 셀의 컴파일 줄에서 `Graph must have an entrypoint` 오류가 납니다.
- 3️⃣ 문서를 가져오기 전에 요약합니다. 빈 문서를 요약하므로 오류 없이 2.4의 `영문 요약:`에 "요약할 글을 보내 달라"는 식의 문장이 나오고, `Ai Message` 칸의 답도 문서와 관계없는 말이 됩니다.
- 4️⃣ 문서를 가져오고, Subgraph로 요약·번역하고, 요약을 바탕으로 답하는 순서로 잇습니다.

</details>

### 2.4 실행하기

In [ ]:
parent_graph_result = app.invoke(
    {
        "messages": [HumanMessage(content="문서의 핵심 내용이 뭐야?")],
        "document": "",
        "summary": "",
    }
)

print(f"영문 요약: {parent_graph_result['summary']}")
results = parent_graph_result["messages"]
for m in results:
    m.pretty_print()

**성공 기준**

- 🟢 `영문 요약:` 뒤에 영어 한 문장이 나온다
- 🟢 `Ai Message` 칸에 문서의 핵심(LLM의 등장으로 AI가 빠르게 발전했고, 글쓰기·번역·코딩 같은 일을 한다)을 담은 한국어 답이 나온다
- ⛔ 오류가 나거나 결과가 위와 다르다 → 2.1~2.3의 성공 기준에서 같은 증상을 찾아 그 셀의 보기를 다시 고르고, 그 셀부터 다시 실행합니다

**볼 지점**
- `영문 요약:`은 Subgraph가 만든 값입니다. 원문은 한국어 두 문장이지만, Subgraph 안에서 요약 → 번역을 거쳐 영어 한 문장이 되었습니다.
- 부모 그래프는 Subgraph 안에 노드가 몇 개 있는지 모릅니다. `summarize_document` 노드 하나로 보일 뿐입니다. 같은 요약·번역 흐름이 다른 그래프에서도 필요하면 `summary_subgraph`를 그대로 가져다 씁니다.
- `Ai Message` 칸의 답은 영문 요약을 바탕으로 만든 답입니다. 원문 대신 요약을 넘겼으므로, 원문의 세부 내용은 답에서 빠질 수 있습니다.

## 3. MemorySaver로 대화 이어 가기

간단한 챗봇 그래프를 만들고 체크포인터 `MemorySaver`를 붙입니다. 같은 `thread_id`로 두 번 호출해, 두 번째 호출이 첫 호출에서 들은 이름을 기억하는지 봅니다.

결과는 맨 아래 "MemorySaver로 컴파일하고 실행하기" 셀에서 나옵니다.

준비 코드입니다. 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 실행만 하면 됩니다.

In [ ]:
import os
import sqlite3
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- raw 기준. 솔루션과의 차이는 주석 처리된 `# mem_app = builder.compile()` 한 줄과 time-travel 부분(다음 장)입니다.
- 이 장은 파일의 앞부분(MemorySaver)까지만 씁니다. SqliteSaver 이후는 patch로 지우고 다음 장에서 다룹니다.
- LLM 호출 2회.
- 원본은 마지막 답만 `print(...content)`로 출력합니다. 노트북은 결과 변수를 그래프가 드러나는 이름(`memory_graph_result`)으로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).

</details>

### 3.1 State와 chatbot 노드

**✅ system 메시지와 State의 messages를 함께 LLM에 전달해 주세요.**

- 🟢 3.3에서 친절한 말투의 답이 출력된다
- ⛔ 3.3에서 `NotImplementedError`나 `ValueError`가 나거나, 이름을 모르고 인사만 한다 → 다른 보기를 고릅니다

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content="너는 친절한 AI 어시스턴트야.")
    # ✅ CHOOSE_HERE: system 메시지와 State의 messages를 함께 LLM에 전달해 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # response = llm.invoke([system, state["messages"]])   # 1️⃣ 시스템 메시지와 메시지 목록을 한 목록에 나란히
    # return {"messages": [response]}                      # 1️⃣
    #
    # response = llm.invoke([system] + state["messages"])  # 2️⃣ 시스템 메시지 뒤에 메시지 목록을 이어 붙임
    # return {"messages": [response]}                      # 2️⃣
    #
    # response = llm.invoke([system])                      # 3️⃣ 시스템 메시지만 넘김
    # return {"messages": [response]}                      # 3️⃣
    #
    # response = llm.invoke(system + state["messages"])    # 4️⃣ 시스템 메시지에 메시지 목록을 바로 더함
    # return {"messages": [response]}                      # 4️⃣

<details><summary>정답 보기</summary>

`chatbot` 노드는 역할을 정한 시스템 메시지와 지금까지 쌓인 대화(`messages`)를 함께 모델에 보내고, 받은 답을 `messages`에 더해 돌려줘야 합니다. 체크포인터가 있으면 앞 호출의 메시지도 `messages`에 들어 있습니다.

**정답: 2️⃣**

- 1️⃣ 목록 안에 목록이 들어갑니다. 모델이 메시지로 읽지 못해 3.3에서 `NotImplementedError: Message as a sequence must be (role string, template)` 오류가 납니다.
- 2️⃣ 시스템 메시지 하나짜리 목록 뒤에 지금까지의 대화를 이어 붙여 모델에 보내고, 답을 `messages`에 더합니다.
- 3️⃣ 대화를 넘기지 않아 모델은 질문도 이름도 모릅니다. 오류 없이 3.3에서 "무엇을 도와드릴까요?" 같은 인사만 나옵니다.
- 4️⃣ 메시지와 목록을 `+`로 더하면 목록이 아니라 다른 객체(프롬프트 템플릿)가 됩니다. 3.3에서 `ValueError: Invalid input type` 오류가 납니다.

</details>

### 3.2 그래프 연결

**✅ chatbot 노드와 START/END 엣지를 연결해 주세요.**

- 🟢 이 셀이 오류 없이 끝나고 `<langgraph.graph.state.StateGraph at 0x…>` 한 줄이 나온다
- ⛔ 이 셀이나 3.3의 컴파일 줄에서 `START cannot be an end node`, `Graph must have an entrypoint` 같은 오류가 난다 → 다른 보기를 고릅니다

In [ ]:
builder = StateGraph(State)
# ✅ CHOOSE_HERE: chatbot 노드와 START/END 엣지를 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# builder.add_edge(START, "chatbot")    # 1️⃣ 엣지만 잇기
# builder.add_edge("chatbot", END)      # 1️⃣
#
# builder.add_node("chatbot", chatbot)  # 2️⃣ 노드 추가 + chatbot→END
# builder.add_edge("chatbot", END)      # 2️⃣
#
# builder.add_node("chatbot", chatbot)  # 3️⃣ 노드 추가 + START→chatbot→START
# builder.add_edge(START, "chatbot")    # 3️⃣
# builder.add_edge("chatbot", START)    # 3️⃣
#
# builder.add_node("chatbot", chatbot)  # 4️⃣ 노드 추가 + START→chatbot→END
# builder.add_edge(START, "chatbot")    # 4️⃣
# builder.add_edge("chatbot", END)      # 4️⃣

<details><summary>정답 보기</summary>

노드가 하나뿐인 그래프입니다. `chatbot` 노드를 등록하고, 시작하면 chatbot으로 가고 chatbot이 끝나면 그래프도 끝나게 잇습니다.

**정답: 4️⃣**

- 1️⃣ 노드를 등록하지 않고 엣지만 이었습니다. 이 셀은 넘어가지만 3.3의 컴파일 줄에서 `Found edge starting at unknown node 'chatbot'` 오류가 납니다.
- 2️⃣ START에서 chatbot으로 가는 엣지가 없습니다. 3.3의 컴파일 줄에서 `Graph must have an entrypoint` 오류가 납니다.
- 3️⃣ 끝(END) 대신 시작점(START)으로 돌아가게 했습니다. START로 들어가는 엣지는 만들 수 없어 이 셀에서 바로 `ValueError: START cannot be an end node` 오류가 납니다.
- 4️⃣ `chatbot` 함수를 같은 이름의 노드로 등록하고 START와 END 사이에 잇습니다.

</details>

### 3.3 MemorySaver로 컴파일하고 실행하기

**✅ MemorySaver 체크포인터로 그래프를 컴파일해 주세요.**

- 🟢 `=== 1. MemorySaver (휘발성) ===` 아래 답에 "홍길동"이 나온다
- ⛔ 이름을 모른다고 답하거나 `TypeError: Invalid checkpointer provided`가 난다 → 다른 보기를 고릅니다

In [ ]:
# ── MemorySaver vs SqliteSaver ────────────────────────────────────────────────
#
# MemorySaver  : 프로세스 메모리에 저장 → 프로그램 재시작 시 기록 사라짐 (개발/데모용)
# SqliteSaver  : SQLite 파일에 저장   → 재시작해도 기록 유지 (로컬 프로토타입용)
# PostgresSaver: PostgreSQL에 저장    → 실제 서비스용 (langgraph-checkpoint-postgres 패키지)

print("=== 1. MemorySaver (휘발성) ===")
# ✅ CHOOSE_HERE: MemorySaver 체크포인터로 그래프를 컴파일해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# mem_app = builder.compile(checkpointer=MemorySaver())  # 1️⃣ MemorySaver를 만들어 체크포인터로 넘김
# mem_app = builder.compile()                            # 2️⃣ 체크포인터 없이 컴파일
# mem_app = builder.compile(checkpointer=MemorySaver)    # 3️⃣ 괄호 없이 MemorySaver를 넘김
# mem_app = builder.compile(store=MemorySaver())         # 4️⃣ MemorySaver를 store 자리에 넘김
config = {"configurable": {"thread_id": "demo"}}

mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름은 홍길동이야.")]}, config=config
)
memory_graph_result = mem_app.invoke(
    {"messages": [HumanMessage(content="내 이름이 뭐야?")]}, config=config
)
results = memory_graph_result["messages"]
for m in results:
    m.pretty_print()

<details><summary>정답 보기</summary>

그래프를 컴파일할 때 체크포인터를 넘기면, 호출이 끝날 때마다 State가 `thread_id`별로 저장됩니다. 아래 코드는 같은 `config`(thread_id `demo`)로 두 번 호출합니다. 첫 번째는 이름을 알려 주고, 두 번째는 이름을 묻습니다.

**정답: 1️⃣**

- 1️⃣ 메모리 저장소를 하나 만들어 체크포인터로 넘깁니다. 호출이 끝날 때마다 State가 thread `demo`에 저장되어 두 번째 호출이 이름을 압니다.
- 2️⃣ 저장소가 없어 호출마다 대화가 새로 시작합니다. 오류 없이 두 번째 답이 "아직 이름을 알려주지 않으셨어요"처럼 이름을 모릅니다.
- 3️⃣ 괄호가 없어 저장소를 만들지 않고 만드는 방법(클래스)만 넘깁니다. 이 셀에서 `TypeError: Invalid checkpointer provided` 오류가 납니다.
- 4️⃣ 체크포인터 자리가 아니라 장기 기억(store) 자리에 넣었습니다. 대화 기록이 저장되지 않아 오류 없이 두 번째 답이 이름을 모릅니다.

</details>

**볼 지점**
- 두 번째 호출에는 "내 이름이 뭐야?"만 보냈는데 "홍길동"이라고 답합니다. 체크포인터가 첫 호출의 메시지를 thread `demo`에 저장해 두었다가 이어 주었기 때문입니다.
- `MemorySaver`는 이 노트북의 메모리에 저장합니다. 커널을 다시 시작하면 기록이 사라집니다. 다음 장의 `SqliteSaver`는 파일에 저장합니다.

## 4. SqliteSaver로 저장하고 되돌아가기

같은 챗봇 그래프를 이번에는 `SqliteSaver`로 컴파일합니다. 대화 기록이 노트북과 같은 폴더의 `checkpoints.db` 파일에 저장되므로 커널을 다시 시작해도 남습니다.

그다음 저장된 체크포인트 목록을 보고, 그중 한 시점으로 되돌아가 다른 질문으로 다시 실행합니다(Time-Travel). 체크포인트는 그래프가 한 단계를 지날 때마다 하나씩 쌓이는 State 사진입니다.

코드는 완성되어 있습니다. 먼저 차례로 실행하고, 마지막 셀의 `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행합니다.

이 장의 준비 코드입니다. 도구를 불러오고, 앞 장에서 만든 것과 같은 State·chatbot 노드·그래프 연결을 완성된 코드로 다시 준비합니다. 펼치지 않고 실행만 합니다.

In [ ]:
import os
import sqlite3
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


class State(TypedDict):
    messages: Annotated[list, add_messages]


def chatbot(state: State) -> dict:
    system = SystemMessage(content="너는 친절한 AI 어시스턴트야.")
    return {"messages": [llm.invoke([system] + state["messages"])]}


builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END);

<details><summary>강사용 메모</summary>

- patch(노트북에서만): ① `DB_PATH`를 `__file__` 기준 대신 노트북 폴더의 `checkpoints.db`로 ② `with sqlite3.connect(...) as conn:` 블록을 `conn = sqlite3.connect(...)`로 풀어 셀 세 개로 나눔(목록 셀과 되돌아가기 셀을 따로 다시 실행할 수 있게) ③ 체크포인트를 앞 3개만이 아니라 번호와 함께 모두 출력, 값이 빈 체크포인트도 출력되게 `cp.values.get("messages", [])` ④ `if len(checkpoints) >= 2:` 조건 제거(두 번 호출하므로 늘 참). 원본 파일은 그대로입니다.
- time-travel 대상: raw는 `checkpoints[0]`(가장 최근이라 되돌아간 효과가 없음), 솔루션은 `checkpoints[-1]`(가장 오래된 것)이고 질문이 "독서 말고 다른 취미도 있어?"라 기억 여부를 가를 수 없습니다. raw 질문을 두고 번호를 CHANGE_HERE로 바꾸게 했습니다(사용자 결정 8).
- 누적: thread_id가 `persistent_demo`로 고정이라 "SqliteSaver로 저장하기" 셀을 다시 실행하면 같은 thread에 기록이 쌓입니다(파일은 커널을 다시 켜도 남음). 되돌아가기 셀을 실행할 때마다 갈래(fork) 체크포인트도 생깁니다. 처음부터 다시 보려면 `checkpoints.db`를 지우거나 thread_id를 바꿉니다.
- `checkpoints.db`는 `.gitignore`의 `*.db`에 걸려 커밋되지 않습니다. 수업이 끝나면 지웁니다.
- 원본은 마지막 답만 `print(...content)`로 출력합니다. 노트북은 결과 변수를 그래프가 드러나는 이름(`sqlite_graph_result`, `time_travel_result`)으로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).

</details>

### 4.1 SqliteSaver로 저장하기

In [ ]:
print("=== 2. SqliteSaver (영속성) ===")
DB_PATH = "checkpoints.db"  # 노트북과 같은 폴더에 만든다

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
# SqliteSaver 체크포인터로 컴파일한다
sqlite_app = builder.compile(checkpointer=SqliteSaver(conn))
config_sql = {"configurable": {"thread_id": "persistent_demo"}}

sqlite_app.invoke(
    {"messages": [HumanMessage(content="내 취미는 독서야.")]}, config=config_sql
)
sqlite_graph_result = sqlite_app.invoke(
    {"messages": [HumanMessage(content="내 취미가 뭐야?")]}, config=config_sql
)
results = sqlite_graph_result["messages"]
for m in results:
    m.pretty_print()

**성공 기준**

- 🟢 `=== 2. SqliteSaver (영속성) ===` 아래 답에 "독서"가 나온다

### 4.2 체크포인트 목록 보기

In [ ]:
# 저장된 체크포인트 목록 확인 (0번이 가장 최근)
checkpoints = list(sqlite_app.get_state_history(config_sql))
print(f"저장된 체크포인트 수: {len(checkpoints)}")
for i, cp in enumerate(checkpoints):
    print(
        f"  [{i}] {cp.config['configurable']['checkpoint_id'][:8]}... | 메시지 수: {len(cp.values.get('messages', []))}"
    )

**성공 기준**

- 🟢 `저장된 체크포인트 수:`와 함께 `[0]`부터 번호가 붙은 목록이 나오고, 처음 실행하면 번호가 커질수록 메시지 수가 같거나 줄어든다
- 🟢 여러 번 실행해도 맨 아래 번호의 메시지 수는 0이다

### 4.3 과거 체크포인트로 되돌아가기

**✍️ CHANGE_HERE: 되돌아갈 체크포인트 번호를 골라 봅니다**

먼저 그대로 실행합니다. 0번은 가장 최근 체크포인트라 되돌아간 것이 아니므로, AI가 취미를 "독서"라고 답합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `0`을 바로 위 목록에서 **메시지 수가 0인 번호**(취미를 말하기 전 시점)로 바꾸고 이 셀만 다시 실행합니다. 목록 셀은 다시 실행하지 않아도 됩니다.

In [ ]:
# Time-Travel: 특정 체크포인트로 되돌아가 다른 흐름으로 재실행
# ✍️ CHANGE_HERE: 되돌아갈 체크포인트 번호를 골라 봅니다
past_config = checkpoints[0].config
print("=== Time-Travel: 과거 시점에서 다른 질문으로 재실행 ===")
time_travel_result = sqlite_app.invoke(
    {"messages": [HumanMessage(content="내 취미가 뭐라고 했더라?")]},
    config=past_config,
)
results = time_travel_result["messages"]
for m in results:
    m.pretty_print()

**성공 기준**

- 🟢 그대로 실행하면(0번) AI가 취미를 "독서"라고 답한다
- 🟢 메시지 수가 0인 번호로 바꾸면 AI가 취미를 모른다고 답한다
- ⛔ `IndexError`가 난다 → 목록에 없는 번호를 넣었습니다. 목록의 번호 중에서 고릅니다
- ⛔ 바꿨는데도 "독서"라고 답한다 → 메시지 수가 1 이상인 번호를 골랐습니다. 목록을 다시 봅니다

**볼 지점**
- 첫 셀을 다시 실행하면 같은 thread(`persistent_demo`)에 기록이 계속 쌓여 체크포인트 수가 늘어납니다. 기록이 파일에 남는다는 증거입니다. 커널을 다시 시작해도 마찬가지입니다.
- 되돌아간 시점에서 다시 실행하면 원래 기록은 그대로 두고 그 시점에서 새 갈래가 생깁니다. 되돌아가기 셀을 실행한 뒤 목록 셀을 다시 실행하면 체크포인트가 늘어난 것을 볼 수 있습니다.
- 처음 상태로 다시 해 보고 싶으면 먼저 커널을 다시 시작합니다(파일이 열려 있으면 Windows에서 지워지지 않습니다). 그다음 노트북과 같은 폴더의 `checkpoints.db`와, 있으면 `checkpoints.db-wal`·`checkpoints.db-shm`까지 세 파일을 지우고 이 장의 셀을 처음부터 실행합니다.

## 5. Memory Store로 대화를 넘어 기억하기

체크포인터는 thread 하나 안에서만 대화를 이어 줍니다. 이번에는 thread를 넘어 남는 장기 기억 저장소(`InMemoryStore`)를 씁니다. 사용자가 이름이나 취미를 말하면 모델이 답 앞에 `[기억 저장: …]`을 붙이고, 코드는 그 내용을 사용자별 namespace(`("memories", user_id)`)에 저장합니다. 다음 대화에서는 저장된 기억을 시스템 메시지에 넣어 줍니다.

코드는 완성되어 있습니다. 먼저 차례로 끝까지 실행하고, `# ✍️ CHANGE_HERE:` 줄 두 곳을 바꿔 다시 실행합니다.

도구와 모델을 준비하는 코드입니다. 실행만 하면 됩니다.

In [ ]:
import re
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.store.memory import InMemoryStore
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 알려진 결함(루트 AGENTS.md 7절): raw의 compile 줄은 `app = builder.compile()`로 checkpointer·store가 없습니다. 슬라이드 p7과 같은 `builder.compile(checkpointer=MemorySaver(), store=store)`로 patch했습니다(원본 파일은 그대로, 사용자 결정 8).
- 그런데 `chatbot`이 전역 변수 `store`를 직접 쓰므로, compile에 `store=`를 넘기든 안 넘기든 화면이 같습니다. LangGraph는 노드 함수가 `store` 인자를 선언할 때(`def chatbot(state, *, store: BaseStore)`)만 compile에 넘긴 store를 넣어 줍니다. 설치된 LangGraph 1.2.12에서 확인했습니다(store 인자를 선언한 노드는 compile(store=...)이면 InMemoryStore, 없으면 None을 받음). 이 점을 CHANGE_HERE 2의 소재로 씁니다. 노드 코드를 고치지는 않았습니다.
- store 실험 순서(2026-09-29 리뷰 반영): `user_002`로 바꾼 상태에서는 세션 2가 원래 기억이 없어 store를 빼도 차이가 드러나지 않으므로, 사용자를 `user_001`로 되돌린 뒤 store를 빼게 했습니다. LLM 없는 가짜 모델로 네 조합(user_001/002 × store 있음/없음)을 돌려, 세션 2가 기억하는지는 사용자에만 달려 있고 store 유무와 관계없음을 확인했습니다.
- 저장은 모델이 `[기억 저장: …]` 형식(콜론 뒤 공백 하나)을 지켜야만 됩니다. 사내 모델의 형식 준수율은 미확인입니다.
- `key = f"fact_{len(memories) + i}"`는 기억이 10개(search 기본 limit)를 넘으면 덮어쓸 수 있습니다. 수업 흐름에서는 생기지 않습니다.

</details>

### 5.1 장기 기억 Store 만들기

In [ ]:
# ── Checkpointer vs Store ─────────────────────────────────────────────────────
#
# Checkpointer (MemorySaver 등)
#   - thread(세션) 단위로 State를 저장
#   - 같은 thread 안에서 대화 흐름을 이어감 (단기 기억)
#
# Store (InMemoryStore 등)
#   - thread를 넘어 여러 세션에서 공유되는 장기 기억
#   - 사용자 프로필, 선호도, 중요 사실 등을 저장
#   - namespace (tuple)로 데이터를 논리적으로 분리한다

# 장기 기억을 저장할 Store를 만든다
store = InMemoryStore()

### 5.2 기억을 읽고 쓰는 chatbot 노드

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    user_id: str


def chatbot(state: State) -> dict:
    user_id = state.get("user_id", "anonymous")

    # Store에서 이 사용자의 장기 기억 조회
    namespace = ("memories", user_id)
    memories = store.search(namespace)
    memory_text = (
        "\n".join(f"- {m.value['fact']}" for m in memories) if memories else "없음"
    )

    system_prompt = f"""너는 친절한 AI 어시스턴트야.

사용자에 대해 알고 있는 정보:
{memory_text}

대화 중 사용자의 중요한 정보(이름, 취미, 직업 등)가 나오면,
"[기억 저장: <내용>]" 형식으로 메시지 앞에 표시해줘."""

    response = llm.invoke([SystemMessage(content=system_prompt)] + state["messages"])

    # LLM이 기억할 내용을 표시했으면 Store에 저장
    content = response.content
    if "[기억 저장:" in content:
        facts = re.findall(r"\[기억 저장: (.+?)\]", content)
        for i, fact in enumerate(facts):
            key = f"fact_{len(memories) + i}"
            store.put(namespace, key, {"fact": fact})

    return {"messages": [response]}

### 5.3 그래프 연결과 컴파일

**✍️ CHANGE_HERE: compile에 넘기는 store를 빼 봅니다**

이 줄은 아래 "두 세션으로 대화하기"의 CHANGE_HERE를 해 본 **다음에** 바꿉니다. 먼저 그 셀에서 바꾼 `"user_002"`를 `"user_001"`로 되돌립니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄에서 `, store=store`를 지워 `builder.compile(checkpointer=MemorySaver())`로 만들고, 이 장의 "장기 기억 Store 만들기" 셀부터 차례로 다시 실행합니다. 세션 2가 여전히 이름(이지수)과 취미(피아노)를 아는지 봅니다.

In [ ]:
builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
# MemorySaver 체크포인터와 Store를 함께 넘겨 컴파일한다
# ✍️ CHANGE_HERE: compile에 넘기는 store를 빼 봅니다
app = builder.compile(checkpointer=MemorySaver(), store=store)

### 5.4 대화 함수

In [ ]:
def chat(user_id: str, thread_id: str, message: str) -> str:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke(
        {"messages": [HumanMessage(content=message)], "user_id": user_id},
        config=config,
    )
    return result["messages"][-1].content

### 5.5 두 세션으로 대화하기

**✍️ CHANGE_HERE: 세션 2의 사용자를 바꿔 봅니다**

처음에는 바꾸지 않고 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `"user_001"`을 `"user_002"`로 바꿉니다. 바꾼 뒤에는 이 장의 "장기 기억 Store 만들기" 셀부터 차례로 다시 실행합니다(대화 기록과 기억을 처음부터 다시 쌓기 위해서입니다).

In [ ]:
print("=== 세션 1: 정보 입력 ===")
print(chat("user_001", "session_1", "안녕! 나는 이지수야. 취미는 피아노 연주야."))

print("\n=== 세션 2: 새 세션에서도 기억 유지 ===")
# ✍️ CHANGE_HERE: 세션 2의 사용자를 바꿔 봅니다
print(chat("user_001", "session_2", "내 이름이랑 취미 알아?"))

print("\n=== Store에 저장된 장기 기억 ===")
memories = store.search(("memories", "user_001"))
for m in memories:
    print(f"  [{m.key}] {m.value['fact']}")

**성공 기준**

- 🟢 세션 1의 답 앞에 `[기억 저장: …]`이 보이고, 세션 2(새 thread)의 답에 "이지수"와 "피아노"가 나온다
- 🟢 `Store에 저장된 장기 기억` 아래에 `[fact_0] …`처럼 기억이 한 줄 이상 나온다
- 🟢 (사용자를 바꾼 뒤) `user_002`의 세션 2 답은 이름과 취미를 모르고, 맨 아래 기억 목록에는 여전히 `user_001`의 기억이 나온다
- 🟢 (사용자를 `user_001`로 되돌리고 compile에서 store를 뺀 뒤) 세션 2의 답에 여전히 "이지수"와 "피아노"가 나오고, 맨 아래 기억 목록에도 기억이 나온다
- ⛔ 기억 목록이 비어 있고 세션 2가 이름을 모른다 → 모델이 `[기억 저장: …]` 형식을 지키지 않았습니다. "장기 기억 Store 만들기" 셀부터 다시 실행합니다

**볼 지점**
- 세션 2는 새 thread(`session_2`)라 체크포인터에는 세션 1의 대화가 없습니다. 그런데도 이름과 취미를 압니다. 시스템 메시지에 Store의 기억이 들어갔기 때문입니다.
- 기억은 `("memories", user_id)` namespace에 저장됩니다. 사용자를 `user_002`로 바꾸면 다른 namespace를 찾으므로 `user_001`의 기억을 보지 못합니다.
- `[기억 저장: …]` 표시는 사용자에게 보이는 답에도 그대로 남습니다. 저장이 모델의 형식 준수에 달려 있다는 점도 이 코드의 약점입니다.
- compile에서 `store=store`를 빼도 세션 2는 여전히 이름과 취미를 압니다. 왜 그런지는 "기억을 읽고 쓰는 chatbot 노드" 셀에서 `store`를 어떻게 쓰는지 보면 알 수 있습니다. 모르겠으면 그 셀을 AI에게 붙여 넣고 물어봅니다.

## 6. 장점과 단점을 동시에 정리하기

한 주제(Remote Work)의 장점과 단점을 두 노드가 동시에 정리하고, 세 번째 노드가 둘을 모아 한 문장 결론을 냅니다. START에서 두 노드로 갈라지는 것이 fan-out, 두 노드가 한 노드로 모이는 것이 fan-in입니다.

결과는 맨 아래 "실행하기" 셀에서 나옵니다.

이 장의 준비 코드입니다. 앞 장들과 같이 도구와 모델을 준비합니다. 펼치지 않고 실행만 합니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- raw 기준. raw의 디버그 출력 `print(state['cons'], state['pros'])`(장단점 전문을 한 번 더 출력)는 뺐고, fan-in 설명 강사 메모 주석(raw 54–59행)은 솔루션의 짧은 주석으로 바꿨습니다. 메모 요지: summary로 들어오는 엣지가 하나면 그 노드가 끝나자마자 summary가 시작되지만, 두 개면 두 노드가 모두 끝나야 summary가 실행된다. 다만 설치된 LangGraph 1.2.12에서는 pros·cons가 같은 단계에서 함께 끝나므로 summary로 가는 엣지가 하나여도 summary는 두 결과를 모두 받습니다(LLM 없는 가짜 모델로 확인, 2026-09-29). 결론이 한쪽만 말하는 것은 START에서 가는 엣지가 빠졌을 때라서 ⛔를 그렇게 적었습니다.
- 19행 주석의 "append"는 두 노드가 서로 다른 필드에 쓰는 동작과 맞지 않아 "자기 필드(pros, cons)에 결과를 쓰고"로 바꿨습니다(2026-09-29 리뷰 반영).
- 이 장은 파일의 앞부분(패턴 1)만 씁니다. 패턴 2(Send)는 patch로 지우고 다음 장에서 다룹니다.
- 직렬(pros → cons → summary)로 이어도 출력이 거의 같아 **화면만으로는 병렬 여부를 가를 수 없습니다.** 성공 기준에 "정답 보기와 엣지 비교"를 넣었습니다.
- LLM 호출 3회.

</details>

### 6.1 장점·단점·결론 노드

In [ ]:
# ── 패턴 1: 정적 병렬 (Fan-out → Fan-in) ────────────────────────────────────
# 고정된 수의 노드를 병렬로 실행할 때 사용한다.
# 각 분기가 자기 필드(pros, cons)에 결과를 쓰고, 이후 하나의 노드가 합쳐서 처리한다.


class StaticState(TypedDict):
    topic: str
    pros: str
    cons: str
    final: str


def pros_node(state: StaticState) -> dict:
    result = llm.invoke(f"{state['topic']}의 장점 3가지를 간결하게 나열해줘.")
    return {"pros": result.content}


def cons_node(state: StaticState) -> dict:
    result = llm.invoke(f"{state['topic']}의 단점 3가지를 간결하게 나열해줘.")
    return {"cons": result.content}


def summary_node(state: StaticState) -> dict:
    result = llm.invoke(
        f"장점:\n{state['pros']}\n\n단점:\n{state['cons']}\n\n위 내용을 바탕으로 한 문장 결론을 내려줘."
    )
    return {"final": result.content}

### 6.2 병렬로 연결하기

**✅ START에서 pros/cons를 병렬로 실행하고 summary에서 합쳐 END로 이어 주세요.**

- 🟢 6.3에서 `장점:`, `단점:`, `결론:`이 차례로 나오고, 결론이 장점과 단점을 함께 말한다
- ⛔ `단점:`이나 `결론:` 뒤가 비었거나, 고른 보기가 `pros` → `cons` → `summary`처럼 차례로 잇는다 → 다른 보기를 고릅니다

In [ ]:
static_builder = StateGraph(StaticState)
static_builder.add_node("pros", pros_node)
static_builder.add_node("cons", cons_node)
static_builder.add_node("summary", summary_node)

# ✅ CHOOSE_HERE: START에서 pros/cons를 병렬로 실행하고 summary에서 합쳐 END로 이어 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# static_builder.add_edge(START, "pros")      # 1️⃣ START→pros→cons→summary→END
# static_builder.add_edge("pros", "cons")     # 1️⃣
# static_builder.add_edge("cons", "summary")  # 1️⃣
# static_builder.add_edge("summary", END)     # 1️⃣
#
# static_builder.add_edge(START, "pros")      # 2️⃣ START→pros, pros·cons→summary→END
# static_builder.add_edge("pros", "summary")  # 2️⃣
# static_builder.add_edge("cons", "summary")  # 2️⃣
# static_builder.add_edge("summary", END)     # 2️⃣
#
# static_builder.add_edge(START, "pros")      # 3️⃣ START→pros·cons, pros·cons→summary→END
# static_builder.add_edge(START, "cons")      # 3️⃣
# static_builder.add_edge("pros", "summary")  # 3️⃣
# static_builder.add_edge("cons", "summary")  # 3️⃣
# static_builder.add_edge("summary", END)     # 3️⃣
#
# static_builder.add_edge(START, "pros")      # 4️⃣ START→pros·cons, 각자 END
# static_builder.add_edge(START, "cons")      # 4️⃣
# static_builder.add_edge("pros", END)        # 4️⃣
# static_builder.add_edge("cons", END)        # 4️⃣
# static_builder.add_edge("summary", END)     # 4️⃣
static_app = static_builder.compile()

<details><summary>정답 보기</summary>

노드 세 개(`pros`, `cons`, `summary`)는 등록되어 있습니다. START에서 `pros`와 `cons`가 함께 시작하고, 두 노드가 모두 끝나면 `summary`가 결론을 내고, 그다음 그래프가 끝나게 잇습니다. 보기 아래의 `static_app = static_builder.compile()` 줄이 그래프를 컴파일합니다.

**정답: 3️⃣**

- 1️⃣ 한 줄로 차례로 이었습니다. 오류 없이 장점·단점·결론이 모두 나와 출력만 보면 정답과 같지만, `cons`는 `pros`가 끝난 뒤에야 시작합니다. 병렬이 아닙니다.
- 2️⃣ START에서 `cons`로 가는 엣지가 없어 `cons`가 실행되지 않습니다. 오류 없이 `단점:` 뒤에 `...`만 나오고 결론이 장점만 말합니다.
- 3️⃣ START에서 두 노드로 갈라져 동시에 시작하고(fan-out), 두 노드가 모두 끝나면 `summary`에서 모입니다(fan-in).
- 4️⃣ `summary`로 들어오는 엣지가 없어 `summary`가 실행되지 않습니다. 오류 없이 장점·단점은 나오지만 `결론:` 뒤가 비어 있습니다.

</details>

### 6.3 실행하기

In [ ]:
print("=== 정적 병렬 (Fan-out → Fan-in) ===")
result = static_app.invoke(
    {"topic": "Remote Work", "pros": "", "cons": "", "final": ""}
)
print(f"장점: {result['pros'][:80]}...")
print(f"단점: {result['cons'][:80]}...")
print(f"결론: {result['final']}")
print()

**볼 지점**
- `장점:`과 `단점:`은 앞 80자만 출력합니다. 두 노드는 서로의 결과를 보지 않고 각자 주제만 받아 정리합니다. 그래서 동시에 실행할 수 있습니다.
- `결론:`은 두 결과를 모두 받은 `summary` 노드가 냅니다. `summary`로 들어오는 엣지가 두 개이므로 두 노드가 모두 끝난 뒤에 실행됩니다.
- 두 노드는 각자 `pros`, `cons`라는 다른 필드에 씁니다. 같은 필드에 쓰지 않으므로 합치는 규칙(reducer)이 필요 없습니다.

## 7. 질문 수만큼 동시에 답하기

이번에는 질문 목록을 받아, 질문 수만큼 worker를 만들어 동시에 답하게 합니다. 질문이 몇 개일지 그래프를 만들 때는 모르므로, 엣지 대신 `Send`로 실행할 때 작업을 만듭니다. `fan_out` 함수가 질문마다 `Send("worker", {...})`를 하나씩 만들어 목록으로 돌려주면, LangGraph가 그 수만큼 `worker` 노드를 병렬로 실행합니다.

worker마다 결과를 `results` 목록에 더합니다. 여러 worker가 같은 필드에 쓰므로, State의 `results`에는 두 목록을 이어 붙이는 규칙(`Annotated[..., lambda a, b: a + b]`)이 붙어 있습니다.

결과는 맨 아래 "실행하기" 셀에서 나옵니다.

준비 코드는 앞 장과 같습니다. 실행만 하면 됩니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Send
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 사용자 결정 9: `fan_out`(raw 94행 TODO)과 동적 연결(raw 111행 TODO)을 과제로 올렸습니다. compile TODO(114행)는 채운 채 뒀습니다.
- raw의 강사 메모 주석(95–100행 for문/리스트 컴프리헨션 설명, 109–110행 기존 간선 방식, 118–119행)은 셀에서 뺐습니다. for문 풀이가 필요한 수강생에게는 말로 보충합니다: `temp = []` → `for q in state['queries']: temp.append(Send('worker', {'query': q, 'results': []}))` → `return temp`와 리스트 컴프리헨션이 같은 일을 한다.
- 결과 줄의 순서는 대개 질문 순서와 같지만, 보장된다고 확인하지는 않았습니다(미확인).

</details>

### 7.1 State와 worker 노드

In [ ]:
# ── 패턴 2: 동적 병렬 (Send API) ────────────────────────────────────────────
# 실행 시점에 작업 수가 결정될 때 사용한다.
# Send(노드명, state)를 리스트로 반환하면 LangGraph가 해당 노드를 병렬로 실행한다.


class DynamicState(TypedDict):
    queries: list[str]
    results: Annotated[list[str], lambda a, b: a + b]  # 결과를 누적


class WorkerState(TypedDict):
    query: str
    results: Annotated[list[str], lambda a, b: a + b]


def worker_node(state: WorkerState) -> dict:
    """각 쿼리를 독립적으로 처리하는 워커 노드."""
    result = llm.invoke(f'"{state["query"]}"에 대해 한 문장으로 답해줘.')
    return {"results": [f"[{state['query']}] {result.content}"]}

### 7.2 fan_out 함수

**✅ queries 수만큼 Send('worker', ...) 목록을 만들어 주세요.**

- 🟢 7.4에서 질문 수만큼 `[질문] 답` 줄이 나온다
- ⛔ 7.4에서 `KeyError: 'query'`가 나거나 줄이 하나만 나온다 → 다른 보기를 고릅니다

In [ ]:
def fan_out(state: DynamicState) -> list[Send]:
    """쿼리 수만큼 worker_node를 병렬로 생성해 보낸다."""
    # ✅ CHOOSE_HERE: queries 수만큼 Send('worker', ...) 목록을 만들어 주세요.
    # (같은 번호의 줄을 모두 선택하고 Ctrl + /)
    # return Send("worker", {"query": state["queries"], "results": []})  # 1️⃣ 질문 목록 전체를 Send 하나에 담음
    # return [Send("worker", {"query": q, "results": []}) for q in state["queries"]]  # 2️⃣ 질문마다 Send 하나씩, 목록으로
    # return [Send("worker", {"question": q, "results": []}) for q in state["queries"]]  # 3️⃣ 질문을 question 키에 담음
    # return [Send(q, {"query": q, "results": []}) for q in state["queries"]]  # 4️⃣ 질문을 보낼 노드 이름 자리에 넣음

<details><summary>정답 보기</summary>

`fan_out`은 State의 `queries`(질문 목록)를 보고, 질문 하나마다 `worker` 노드로 보낼 `Send`를 하나씩 만들어 목록으로 돌려줘야 합니다. `Send`에 담는 딕셔너리는 worker가 받을 State(`WorkerState`)의 모양을 따릅니다. 이 셀은 함수만 만들고 아무것도 출력하지 않습니다.

**정답: 2️⃣**

- 1️⃣ `Send`를 하나만 만들어 worker가 한 번만 실행됩니다. 오류 없이 7.4에 `[['LangGraph란?', 'RAG란?', 'MCP란?']] …`로 시작하는 줄 하나만 나옵니다.
- 2️⃣ 질문 하나마다 `worker`로 보낼 `Send`를 만들어 목록으로 돌려줍니다. 담는 값은 `WorkerState`의 키(`query`, `results`)에 맞췄습니다.
- 3️⃣ worker는 `state["query"]`를 읽는데 `question`에 담았습니다. 7.4에서 `KeyError: 'query'` 오류가 납니다.
- 4️⃣ `Send`의 첫 값은 보낼 노드 이름입니다. `LangGraph란?` 같은 노드는 없어 LangGraph가 `Ignoring unknown node name` 경고만 남기고 건너뜁니다. 오류 없이 7.4에 결과 줄이 하나도 나오지 않습니다.

</details>

### 7.3 동적으로 연결하기

**✅ START에서 fan_out으로 worker를 동적 생성하고 worker 이후 END로 연결해 주세요.**

- 🟢 이 셀이 오류 없이 끝나고, 7.4에서 질문 세 개의 `[질문] 답` 줄이 나온다
- ⛔ 이 셀에서 `TypeError`나 `Found edge ending at unknown node`가 나거나, 7.4에서 `KeyError: 'query'`가 난다 → 다른 보기를 고릅니다

In [ ]:
dynamic_builder = StateGraph(DynamicState)
dynamic_builder.add_node("worker", worker_node)
# ✅ CHOOSE_HERE: START에서 fan_out으로 worker를 동적 생성하고 worker 이후 END로 연결해 주세요.
# (같은 번호의 줄을 모두 선택하고 Ctrl + /)
# dynamic_builder.add_conditional_edges(START, fan_out, ["worker"])  # 1️⃣ START에서 fan_out으로 조건부 엣지 + worker→END
# dynamic_builder.add_edge("worker", END)                     # 1️⃣
#
# dynamic_builder.add_edge(START, "worker")                   # 2️⃣ START→worker 일반 엣지
# dynamic_builder.add_edge("worker", END)                     # 2️⃣
#
# dynamic_builder.add_conditional_edges(START, fan_out(), ["worker"])  # 3️⃣ fan_out을 불러 그 결과를 넘김
# dynamic_builder.add_edge("worker", END)                     # 3️⃣
#
# dynamic_builder.add_edge(START, fan_out)                    # 4️⃣ add_edge에 fan_out 함수를 넘김
# dynamic_builder.add_edge("worker", END)                     # 4️⃣
dynamic_app = dynamic_builder.compile()

<details><summary>정답 보기</summary>

`worker` 노드는 등록되어 있습니다. 시작하면 `fan_out`이 돌려준 `Send` 목록대로 worker를 만들고, worker가 끝나면 그래프가 끝나게 잇습니다. `Send` 목록을 돌려주는 함수는 조건부 엣지에 넣습니다. 보기 아래의 `dynamic_app = dynamic_builder.compile()` 줄이 그래프를 컴파일합니다.

**정답: 1️⃣**

- 1️⃣ START에서 `fan_out`을 불러 돌려받은 `Send` 목록대로 worker를 만들고, worker가 끝나면 END로 갑니다. 갈 수 있는 노드는 `["worker"]`로 알려 줍니다.
- 2️⃣ `fan_out`을 거치지 않아 worker가 질문 하나 없이 그래프 State 그대로 한 번 실행됩니다. 7.4에서 `KeyError: 'query'` 오류가 납니다.
- 3️⃣ 괄호를 붙이면 State 없이 그 자리에서 함수를 부릅니다. 이 셀에서 `TypeError: fan_out() missing 1 required positional argument: 'state'` 오류가 납니다.
- 4️⃣ `add_edge`의 도착에는 노드 이름만 씁니다. 함수는 노드가 아니라서 이 셀에서 `Found edge ending at unknown node` 오류가 납니다.

</details>

### 7.4 실행하기

**✍️ CHANGE_HERE: 질문을 하나 더 넣어 봅니다**

처음 실행은 목록을 그대로 둡니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 목록 끝에 질문을 하나 더 넣고(예: `"A2A란?"`) 이 셀만 다시 실행합니다. 따옴표와 쉼표를 지킵니다.

In [ ]:
print("=== 동적 병렬 (Send API) ===")
# ✍️ CHANGE_HERE: 질문을 하나 더 넣어 봅니다
queries = ["LangGraph란?", "RAG란?", "MCP란?"]
result2 = dynamic_app.invoke({"queries": queries, "results": []})
for r in result2["results"]:
    print(f"  {r}")

**성공 기준**

- 🟢 처음 실행에서 `[LangGraph란?]`, `[RAG란?]`, `[MCP란?]`로 시작하는 줄이 세 줄 나온다
- 🟢 질문을 하나 더 넣으면 줄도 하나 늘어난다(그래프는 그대로인데 worker 수가 바뀐다)
- ⛔ 줄 수가 질문 수보다 적다 → 질문 목록의 따옴표와 쉼표를 다시 봅니다

**볼 지점**
- 그래프에는 `worker` 노드가 하나뿐인데, 질문 수만큼 실행되었습니다. 몇 번 실행할지는 그래프를 만들 때가 아니라 실행할 때 `fan_out`이 정합니다.
- worker들은 동시에 실행되므로, 질문을 늘려도 걸리는 시간은 질문 수만큼 늘지 않습니다.
- 모든 worker가 같은 `results`에 결과를 더했는데도 줄이 모두 남았습니다. 왜 그런지는 "State와 worker 노드" 셀에서 `results`에 붙은 `Annotated[..., lambda a, b: a + b]`를 보면 알 수 있습니다. 모르겠으면 그 셀을 AI에게 붙여 넣고 물어봅니다.

## 8. 정리

LangGraph로 조금 더 큰 그래프를 만드는 방법을 여섯 장에 걸쳐 실행해 보았습니다.

| 장 | 한 일 | 확인한 것 |
|---|---|---|
| 2. Subgraph | 요약·번역 그래프를 부모 노드 안에서 호출했습니다 | State가 달라도 노드 함수가 값을 옮겨 주면 됩니다 |
| 3. MemorySaver | 같은 thread_id로 두 번 호출했습니다 | 체크포인터가 앞 대화를 이어 줍니다 |
| 4. SqliteSaver | 파일에 저장하고 지난 시점으로 되돌아갔습니다 | 기록이 파일에 남고, 과거 체크포인트에서 새 갈래로 다시 실행할 수 있습니다 |
| 5. Memory Store | 새 thread에서도 사용자 정보를 기억하게 했습니다 | 장기 기억은 namespace로 사용자별로 나뉩니다. 노드가 어떤 store를 쓰는지는 코드를 봐야 압니다 |
| 6. Fan-out/Fan-in | 장점·단점을 동시에 정리하고 합쳤습니다 | 병렬인지는 출력이 아니라 엣지를 봐야 알 수 있습니다 |
| 7. Send | 질문 수만큼 worker를 만들어 동시에 답하게 했습니다 | 작업 수를 실행할 때 정하고, 결과는 reducer로 모읍니다 |

보기를 골라 실행하고 값을 바꿔 다시 실행한 뒤, 결과를 성공 기준(🟢/⛔)과 비교하고 정답 보기의 해설로 확인했습니다.